<a href="https://colab.research.google.com/github/patriciomontenegro/conectar_AI/blob/main/mecassist_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CREANDO UN AGENTE DE IA PARA LA EMPRESA MECASSIST

In [14]:
!pip install -q langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.5 MB/s eta 0:00:00


In [16]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('gemini_Api_Key')

In [40]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

In [41]:
#preguntemos algo a la llm
respuesta = llm.invoke("¿sobre que trata el presente trabajo?")

In [42]:
#intenta mostrar una respuesta

respuesta.content

[{'type': 'text',
  'text': 'Para poder decirte de qué trata el trabajo, necesito que me proporciones más información, ya que actualmente no tengo ningún texto o documento cargado.\n\nPor favor, realiza alguna de las siguientes acciones:\n\n1. **Pega el texto** o un resumen del trabajo aquí en el chat.\n2. **Sube el archivo** (si tienes la opción de adjuntar documentos).\n3. **Dime el título, el autor o el tema** del trabajo al que te refieres.\n\n¡En cuanto compartas esa información conmigo, con gusto lo analizaré y te daré una explicación detallada de su contenido!',
  'extras': {'signature': 'EpIOCo8OARFNMg98wpBhxFjd0ieCJoOaR5kSImQnuDxVO/3St3MTFZdniox1wC87ImDWU2q5KKKJzO/dnw+FQpb0Hmtgc6Ff/aCJEjCtZtkVvUDM0SGgY9nQDlANkuQvQFJn/VR9AVajeXeBJxXMmDLeL0ndOzWOGE1QWYXPAhvEYMT2tdPCKqs0g/tdsfA8MAlLcBrt38hUPJLEb65sKAd9RCYaFSUfFq4KvVBGHInD0KtDzXksVhNkqBI4I6pZQflQWJtwiCOdPYYG8rpc92NfDXf+sL++6GTZLKMe2evMQ3ImdhKvuNf8Fugyw7WZYhrkneK1PISi7vywaDYfJItZqKuee56XkSaIPeCxo+LKSUQMV6nsYZ+asSkjUItosmQxUQkRZIAVy

In [44]:
PROMPT_TRIAJE = """
Eres un especialista en triaje de la empresa mecassist para politiccas externas.
Dado el mensaje del usuario, devuelve SÓLO un JSON con:\n
{\n
    "decision": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_TICKET",\n
    "urgencia": "BAJA" | "MEDIANA" | "ALTA",\n
    "campos_faltantes": ["..."]\n
}\n
Reglas:\n
- **AUTO_RESOLVER**: Preguntas claras sobre las reglas o procedimientos descritos en las politicas (Ej.: "¿a que hora abren los días lunes?").\n
- **PEDIR_INFO**: Mensajes imprecisos o sin información para identificar el tema o el contexto (Ej.: "Necesito ayuda con mi vehiculo").\n
- **ABRIR_TICKET**: Solicitudes de excepciones, autorización, aprobación o acceso especial, o cuando el usuario solicita explicitamente abrir un ticket (Ej.: "saqué hace dos días el vehículo del taller y hoy no enciende, que hago?").\n
Analiza el mensaje y decide la acción más adecuada.
"""

In [49]:
from typing import Literal, List, Dict
from pydantic import BaseModel, Field

class TriajeOut(BaseModel):
  decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_TICKET"]
  urgencia: Literal["BAJA", "MEDIANA", "ALTA"]
  campos_faltantes: List[str] = Field(default_factory=list)

In [50]:
from langchain_core.messages import SystemMessage, HumanMessage

chain_de_triaje = llm.with_structured_output(TriajeOut)

def triaje(mensaje:str) -> Dict:
  salida: TriajeOut = chain_de_triaje.invoke(
      [
          SystemMessage(content=PROMPT_TRIAJE),
          HumanMessage(content=mensaje)
      ]
  )

  return salida.model_dump()

In [57]:
mensajes_de_prueba = [
    "¿a que hora abren el día sabado?",
    "¿aseguran la satisfacción del cliente?",
    "¿sus trabajos tienen garantía?",
    "¿que hacen con el vehículo al momento de ingresar al taller?",
    "¿van a ver el mundial de futbol?"
]

for pregunta in mensajes_de_prueba:
  r = triaje(pregunta)
  print(f"{pregunta} -> [r]")

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 33.276858114s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT